<a href="https://colab.research.google.com/github/angadghatode/ISYS2001/blob/main/Module%2003%20-%20Making%20Computers%20Think/lab_ticket_budget_decisions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/kevin-blasiak-curtin/ISYS2001-Archive/blob/main/Module%2003%20-%20Making%20Computers%20Think/lab_ticket_budget_decisions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 Lab Ticket: Budget Decisions

This week your job is not to hand-code a program. It is to write **one complete, detailed prompt** that an AI can turn into a working budget tool in a single go.

In the worksheet you practised giving AI clear intent. Here that is the whole task. The skill being tested is whether you can describe exactly what you want, in enough detail, that one prompt produces the finished tool with no back-and-forth.

## What to submit

1. **Your final prompt**: one complete, self-contained prompt (paste it into the prompt cell below).
2. **The code it produced**: paste the AI-generated program into the code cell and run it, so you can show it works.

Your prompt can be the result of several rounds of refining. What you hand in is the single finished version: the one that would generate the whole tool if someone ran it cold.

## The brief

Design a tool that helps someone make a smart budget decision. What that means, and who it is for, is up to you. Some directions, though you are not limited to these:

- An expense classifier that sorts spending into categories and reacts to each.
- A budget checker that warns when an expense is too large a share of someone's budget.
- A savings goal tracker that gives different feedback depending on progress.
- A purchase advisor that weighs a price against how much money and income the person has.

Pick one or invent your own. The only hard requirement is that the finished program makes genuinely different decisions depending on the numbers it is given.

## How this works

You are going to write a prompt, hand it to an AI, and see what it builds. Your first attempt will not be right. Read what comes back and look for the gaps:

- Did it invent a rule you did not ask for? Your prompt was not precise enough.
- Did it miss a case? Say so explicitly next time.
- Did it ask you follow-up questions? A complete prompt should not need any.

Fold each fix back into the prompt and run it again. Keep going until one single prompt produces the whole working tool in one go. That final version is what you submit.

How detailed does it need to be? Detailed enough that a classmate could paste it cold and get the same working tool. Work out for yourself what that takes.

## Your final prompt

> Paste your complete, finished prompt below (double-click to edit).

```
PROMPT 1
Build a "Purchase Advisor" tool in Python for the terminal that takes a purchase price, current savings, monthly income, and monthly essential expenses, and returns a clear verdict (approve/caution/reject) with reasoning that meaningfully differs based on which factor is driving the risk (e.g. wiping out savings vs. straining thin disposable income), rather than one formula with the wording adjusted, so a $50 purchase, a purchase that's fine on income but drains savings, and one that's fine on savings but eats most of a thin income all produce genuinely different verdicts and explanations.
make it vary a little between what the outcomes could be depending on the price of the product and what the product is but account for edge cases and improbable answers.

--- I was given a 700 code program with this first prompt which was overkill for what I needed.

PROMPT 2
this is too much for what I need, I only need the bare bones up to 200-300 lines of code. Can you condense this all down to a very simplistic approach (MVP) for the python terminal

```

In [5]:
#!/usr/bin/env python3
"""
Purchase Advisor (MVP)
=======================

A simple terminal tool: give it a purchase price, current savings,
monthly income, and monthly essential expenses, and it returns a
verdict -- APPROVE / CAUTION / REJECT -- with a short explanation of
*why*, based on whichever is actually the problem: your savings
buffer or your monthly cash flow.

Usage:
    python3 purchase_advisor.py --price 1200 --savings 8000 --income 4500 --expenses 3200
    python3 purchase_advisor.py                # interactive prompts

Not financial advice -- just a sanity check.
"""

import argparse
import sys

# --- tunable thresholds -----------------------------------------------

TRIVIAL_PRICE = 50          # dollar amount small enough to basically ignore
TARGET_BUFFER_MONTHS = 3    # recommended minimum "emergency fund" size
THIN_MARGIN_RATIO = 0.15    # below this % of income left over = thin
BIG_CHUNK_OF_MARGIN = 3     # purchase costing 3+ months of margin = a lot


def parse_number(raw, allow_negative=False):
    cleaned = raw.strip().replace("$", "").replace(",", "")
    value = float(cleaned)
    if not allow_negative and value < 0:
        raise ValueError("must be zero or positive")
    return value


def get_inputs_from_args(args):
    return args.price, args.savings, args.income, args.expenses, args.item


def get_inputs_interactively():
    print("Purchase Advisor -- interactive mode\n")

    def ask(prompt, allow_negative=False):
        while True:
            raw = input(prompt)
            try:
                return parse_number(raw, allow_negative)
            except ValueError:
                print("  Please enter a valid number.")

    price = ask("Purchase price: $")
    savings = ask("Current savings: $", allow_negative=True)
    income = ask("Monthly income: $")
    expenses = ask("Monthly essential expenses: $")
    item = input("What is it? (optional): ").strip()
    return price, savings, income, expenses, item


def evaluate(price, savings, income, expenses):
    """Return (verdict, headline, reasons: list[str])."""

    disposable = income - expenses
    post_purchase_savings = savings - price
    buffer_after = (post_purchase_savings / expenses) if expenses > 0 else None
    margin_ratio = (disposable / income) if income > 0 else None

    # --- edge case: already spending more than you earn ---------------
    if income > 0 and disposable <= 0:
        if price <= TRIVIAL_PRICE and post_purchase_savings >= 0:
            return (
                "CAUTION",
                "Your expenses already exceed your income -- this small purchase isn't the real issue.",
                [
                    f"Essential expenses exceed income by ${abs(disposable):,.2f}/month, "
                    f"regardless of this purchase.",
                    "The bigger priority is closing that monthly gap, not this specific item.",
                ],
            )
        return (
            "REJECT",
            "Your essential expenses already exceed your income -- this isn't the time to add spending.",
            [
                f"You're short ${abs(disposable):,.2f}/month before this purchase is even considered.",
                "Any extra spending right now comes straight out of savings you're already relying on.",
            ],
        )

    # --- edge case: you can't actually afford it from savings ---------
    if post_purchase_savings < 0:
        return (
            "REJECT",
            "You don't have enough saved to cover this without going into debt.",
            [
                f"It costs ${price:,.2f} but you only have ${savings:,.2f} saved -- "
                f"buying it would leave you ${abs(post_purchase_savings):,.2f} short.",
                f"Your monthly margin (${disposable:,.2f} left after essentials) is healthy, "
                f"but that's separate from being able to afford this right now.",
            ],
        )

    # --- trivial purchase: tiny relative to both income and savings ---
    is_trivial = (
        price <= TRIVIAL_PRICE
        and (income <= 0 or price / income < 0.02)
        and (savings <= 0 or price / savings < 0.02)
    )
    if is_trivial:
        return (
            "APPROVE",
            f"${price:,.2f} barely registers against your finances -- go ahead.",
            ["This is small enough relative to your income and savings that it's not worth overthinking."],
        )

    # --- savings-driven risk: purchase eats into your safety net ------
    if buffer_after is not None and buffer_after < 1:
        return (
            "REJECT",
            "This would gut your savings buffer.",
            [
                f"Savings would drop from ${savings:,.2f} to ${post_purchase_savings:,.2f} -- "
                f"only {buffer_after:.1f} month(s) of expenses left in reserve.",
                f"Your monthly budget itself looks fine "
                f"({'about ' + format(margin_ratio * 100, '.0f') + '% of income left over' if margin_ratio is not None else 'no income given'}), "
                f"but this specific purchase drains your emergency fund.",
            ],
        )

    if buffer_after is not None and buffer_after < TARGET_BUFFER_MONTHS:
        return (
            "CAUTION",
            "Affordable, but it leans hard on your savings buffer.",
            [
                f"After this purchase you'd have {buffer_after:.1f} months of expenses saved "
                f"(recommended minimum is {TARGET_BUFFER_MONTHS}).",
                "Your monthly cash flow isn't the problem -- your safety net gets thinner.",
            ],
        )

    # --- income-driven risk: safety net is fine, but monthly room is thin
    if margin_ratio is not None and margin_ratio < THIN_MARGIN_RATIO and disposable > 0:
        months_of_margin = price / disposable
        if months_of_margin >= BIG_CHUNK_OF_MARGIN:
            return (
                "CAUTION",
                "Your savings can easily absorb this, but your month-to-month room is razor-thin.",
                [
                    f"Only about ${disposable:,.2f}/month ({margin_ratio * 100:.0f}% of income) "
                    f"is left after essentials.",
                    f"This purchase equals roughly {months_of_margin:.1f} months of that thin margin -- "
                    f"savings aren't at risk, but your day-to-day flexibility is limited either way.",
                ],
            )

    # --- otherwise, comfortable -----------------------------------------
    reasons = []
    if buffer_after is not None:
        reasons.append(
            f"Savings stay healthy after this purchase: {buffer_after:.1f} months of expenses in reserve."
        )
    if margin_ratio is not None:
        reasons.append(
            f"Monthly margin stays healthy too: about {margin_ratio * 100:.0f}% of income left after essentials."
        )
    if not reasons:
        reasons.append("Nothing about your numbers suggests a problem here.")
    return ("APPROVE", "This fits comfortably within your savings and your monthly budget.", reasons)


def print_report(item, price, verdict, headline, reasons):
    label = f"'{item.strip()}'" if item and item.strip() else "This purchase"
    print("\n" + "-" * 60)
    print(f"VERDICT: {verdict}")
    print("-" * 60)
    print(f"{label} (${price:,.2f}): {headline}\n")
    print("Why:")
    for r in reasons:
        print(f"  - {r}")
    print("\n(Heuristic sanity check only -- not financial advice.)")
    print("-" * 60 + "\n")


def main():
    parser = argparse.ArgumentParser(description="Purchase Advisor -- quick approve/caution/reject check.")
    parser.add_argument("-p", "--price", type=float)
    parser.add_argument("-s", "--savings", type=float)
    parser.add_argument("-i", "--income", type=float)
    parser.add_argument("-e", "--expenses", type=float)
    parser.add_argument("--item", type=str, default="")
    # parse_known_args ignores extra flags Colab/Jupyter inject on their own
    # (e.g. "-f /root/.../kernel-xxxx.json"), instead of erroring out on them.
    args, _unknown = parser.parse_known_args()

    if None not in (args.price, args.savings, args.income, args.expenses):
        price, savings, income, expenses, item = get_inputs_from_args(args)
    else:
        price, savings, income, expenses, item = get_inputs_interactively()

    if price <= 0:
        print("Error: purchase price must be greater than zero.", file=sys.stderr)
        sys.exit(1)
    if income < 0 or expenses < 0:
        print("Error: income and expenses can't be negative.", file=sys.stderr)
        sys.exit(1)

    verdict, headline, reasons = evaluate(price, savings, income, expenses)
    print_report(item, price, verdict, headline, reasons)


if __name__ == "__main__":
    main()

Purchase Advisor -- interactive mode

Purchase price: $2000
Current savings: $11200
Monthly income: $1200
Monthly essential expenses: $300
What is it? (optional): iPhone

------------------------------------------------------------
VERDICT: APPROVE
------------------------------------------------------------
'iPhone' ($2,000.00): This fits comfortably within your savings and your monthly budget.

Why:
  - Savings stay healthy after this purchase: 30.7 months of expenses in reserve.
  - Monthly margin stays healthy too: about 75% of income left after essentials.

(Heuristic sanity check only -- not financial advice.)
------------------------------------------------------------



## Quick reflection

> Double-click to answer.

1. What did your first prompt miss that you had to add?
2. Which single detail made the biggest difference to the output?
3. Could a classmate run your final prompt cold and get a working tool? How do you know?

## Before you submit

- [ ] Your final prompt is one complete block that needs no follow-up questions.
- [ ] The generated code runs top to bottom with no errors.
- [ ] The program makes different decisions for different inputs.
- [ ] You can explain what every part of your prompt is doing and why.
- [ ] You have downloaded the notebook and submitted it as your Week 3 lab ticket.

This is the first brick in a bigger wall. Over the semester your budget logic grows into a finance tracker, and the prompt-writing skill you practise here is one you will lean on the whole way.